# Per-superclass LaTeX table for selected (Tau, m) configs

Reads every `*.csv` in `analysis/data/resnet50`, picks the rows matching the requested `(Tau, m)` configurations, and emits a `table*` with one block per taxonomic level (O/F/G/S) and one column per superclass.

In [ ]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("data/resnet50")
PROBE = "linprobe"  # "linprobe" or "knn"
TOPK = "top1"  # "top1" or "top5"

# (row label, Tau, m) -- use "-" for a blank cell in the CSV
CONFIGS = [
    ("Frozen Backbone", "-", "-"),
    ("TaxoCon(ours)", "0.7", "0.999"),
]

SUPERCLASS_ORDER = [
    "mollusks",
    "mammals",
    "fishes",
    "insects",
    "birds",
    "reptiles",
    "amphibians",
    "fungi",
    "plants",
]

LEVELS = {  # table block label -> column stem in the CSV
    "O": "val_train_order",
    "F": "val_test_family",
    "G": "val_test_genus",
    "S": "val_test_specific_epithet",
}

files = sorted(DATA_DIR.glob("*.csv"))
print(f"{len(files)} files:", [f.stem for f in files])

In [ ]:
TAU_ZERO_INF_LABEL = "infinite"  # tau=0 and tau=infinity denote the same setting
INF_ALIASES = {"inf", "+inf", "-inf", "infty", "infinity", "infinite", "\u221e"}


def normalize_key(value) -> str:
    """Collapse equivalent numeric labels (1 vs 1.0 vs 1.00) to one string; blanks -> '-'."""
    if pd.isna(value) or str(value).strip() == "":
        return "-"
    try:
        return f"{float(value):g}"
    except ValueError:
        return str(value).strip()


def normalize_tau(value) -> str:
    text = str(value).strip().lower()
    if text in INF_ALIASES:
        return TAU_ZERO_INF_LABEL
    key = normalize_key(value)
    if key in {"0", "-0", "inf", "-inf"}:
        return TAU_ZERO_INF_LABEL
    return key


metric_cols = {label: f"{stem}_{PROBE}_{TOPK}" for label, stem in LEVELS.items()}


def load(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, dtype={"Tau": str, "m": str})
    missing = [c for c in metric_cols.values() if c not in df.columns]
    if missing:
        raise KeyError(f"{missing} not in {path.name}")
    df = df[["Tau", "m", *metric_cols.values()]].copy()
    df["Tau"] = df["Tau"].map(normalize_tau)
    df["m"] = df["m"].map(normalize_key)
    for col in metric_cols.values():
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["source"] = path.stem.split("-")[0].lower()
    return df


long = pd.concat([load(f) for f in files], ignore_index=True)
long.head()

In [ ]:
sources = [s for s in SUPERCLASS_ORDER if s in set(long["source"])]
missing_sources = sorted(set(long["source"]) - set(sources))
if missing_sources:
    print("not in SUPERCLASS_ORDER, skipped:", missing_sources)


def block(level: str) -> pd.DataFrame:
    col = metric_cols[level]
    rows = {}
    for label, tau, m in CONFIGS:
        tau_key, m_key = normalize_tau(tau), normalize_key(m)
        sel = long[(long["Tau"] == tau_key) & (long["m"] == m_key)]
        if sel.empty:
            print(f"warning: no rows for {label} (Tau={tau_key}, m={m_key})")
        rows[label] = sel.groupby("source")[col].mean().reindex(sources) * 100
    return pd.DataFrame(rows).T[sources]


blocks = {level: block(level) for level in LEVELS}
blocks["O"].round(2)

In [ ]:
def fmt(value, best: bool) -> str:
    if pd.isna(value):
        return "--"
    text = f"{value:.2f}"
    return f"\\textbf{{{text}}}" if best else text


def to_latex(blocks: dict[str, pd.DataFrame]) -> str:
    width = max(len(label) for label, _, _ in CONFIGS)
    header = " & ".join(["Metric", "Method", *(s.capitalize() for s in sources)])
    lines = [
        "\\begin{table*}[ht]",
        "\\centering",
        "\\small",
        "\\renewcommand{\\arraystretch}{0.85}",
        "\\resizebox{\\textwidth}{!}{%",
        "\\begin{tabular}{ll" + "c" * len(sources) + "}",
        "\\toprule",
        header + " \\\\",
        "\\midrule",
    ]
    for i, (level, table) in enumerate(blocks.items()):
        if i:
            lines.append("\\midrule")
        lines.append(f"\\multirow{{{len(table)}}}{{*}}{{{level}}}")
        best = table.max(axis=0)
        for label, row in table.iterrows():
            cells = " & ".join(fmt(v, v == best[c]) for c, v in row.items())
            lines.append(f"& {label:<{width}} & {cells} \\\\")
    lines += [
        "\\bottomrule",
        "\\end{tabular}%",
        "}",
        f"\\caption{{Per-superclass {PROBE} {TOPK} accuracy for ResNet50}}",
        "\\label{tab:resnet50_" + PROBE + "}",
        "\\end{table*}",
    ]
    return "\n".join(lines)


latex = to_latex(blocks)
print(latex)

In [ ]:
out = DATA_DIR.parent / f"per_superclass_{PROBE}_{TOPK}.tex"
out.write_text(latex, encoding="utf-8")
print("wrote", out)